<a href="https://colab.research.google.com/github/esraa9895/thesis-code/blob/main/different_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

## Rosenbrock function

In [ ]:
def fun(x):
    return 100 * (x[1] - x[0]**2)**2 + (1 - x[0])**2

def grad(x):
    return np.array([-400*x[0]*(x[1] - x[0]**2) - 2*(1 - x[0]), 200*(x[1] - x[0]**2)])

def hess(x):

    return np.array([[1200*x[0]**2 - 400*x[1] + 2, -400*x[0]], [-400*x[0], 200]])

## 41. EG2(CUTE)

In [ ]:

def fun(x):
    return np.sin(x[0]**2 + x[0] - 1) + 0.5*np.sin(x[1]**2)

def grad(x):
    df_dx0 = (2*x[0] + 1) * np.cos(x[0]**2 + x[0] - 1)
    df_dx1 = x[1] * np.cos(x[1]**2)
    return np.array([df_dx0, df_dx1])

def hess(x):
    d2f_dx02 = - (2*x[0] + 1)**2 * np.sin(x[0]**2 + x[0] - 1) + 2 * np.cos(x[0]**2 + x[0] - 1)
    d2f_dx12 = np.cos(x[1]**2) - 2 * x[1]**2 * np.sin(x[1]**2)
    return np.array([[d2f_dx02, 0],
                     [0, d2f_dx12]])

# Example usage:
x0 = np.array([-0.5, 0])
print("fun:", fun(x0))
print("Gradient:", grad(x0))
print("Hessian:\n", hess(x0))

 Function
0.5*sin(x2**2) - sin(-x1**2 + x1 + 1)

 First Derivative
d/dx1= -(1 - 2*x1)*cos(-x1**2 + x1 + 1)
d/dx2= 1.0*x2*cos(x2**2)

 Second Derivative
d**2/dx1**2= (2*x1 - 1)**2*sin(-x1**2 + x1 + 1) + 2*cos(-x1**2 + x1 + 1)
d**2/dx2**2= -2.0*x2**2*sin(x2**2) + 1.0*cos(x2**2)

## plotting functions

In [ ]:
#plot objective function
def fun_plot(fun,Title, initial):
    x1 = np.arange(-2, 2, .01)
    x2 = np.arange(-2, 2, .01)

    # Creating 2-D grid of features
    X = np.meshgrid(x1, x2)

    fig, ax = plt.subplots()

    Z = fun(X)

    # plots contour lines
    ax.contour(X[0], X[1], Z,100)
    plt.plot(-.5,0,marker='*',markersize =5)
    plt.plot(initial[0],initial[1], color = 'black')
    ax.set_title(Title)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')

In [ ]:
#plot the function by different methods
def plot_3D(fun,initial):
    # x and y axis
    x1 = np.arange(-3, 3, .01)
    x2 = np.arange(-3, 3, .01)

      # Creating 2-D grid of features
    X = np.meshgrid(x1, x2)
    Z = fun(X)

    ax = plt.axes(projection='3d')

    ax.plot_surface(X[0], X[1], Z, cmap='hot', alpha=0.8)
    ax.plot(0.5,0,marker='*', color= 'black')
    plt.plot(initial[0],initial[1],marker='^',markersize =5,color= 'black')
    ax.set_title('3D Contour Plot of rosenbrock function', fontsize=14)
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_zlabel('z', fontsize=12)

In [ ]:
#get accuracy for each method
def accuracy(f):

    return  1- (np.absolute(fun([-.5, 0])-f)) #41. EG2(CUTE)
    #np.absolute(1.0-f)

## get step length in each iteration

In [ ]:
#line_search method used in algorithm
def strong_wolfe(f, g, xk, alpha, pk, c2):
  # typically, c2 = 0.9 when using Newton or quasi-Newton's method.
  #            c2 = 0.1 when using non-linear conjugate gradient method.
  c1 = 1e-4
  return f(xk + alpha * pk) <= f(xk) + c1 * alpha * np.dot(pk.T,g(xk)) and abs(
       np.dot(pk.T,g(xk + alpha * pk))) <= c2 * abs(np.dot(pk.T,g(xk)))

# line-search step len
def step_length(f, g, xk, alpha, pk, c2):

  return interpolation(f, g,
                       lambda alpha: f(xk + alpha * pk),
                       lambda alpha: np.dot(g(xk + alpha * pk), pk),
                       alpha, c2,
                       lambda f, g, alpha, c2: strong_wolfe(f, g, xk, alpha, pk, c2))


def interpolation(f, g, f_alpha, g_alpha, alpha, c2, strong_wolfe_alpha, iters=20):
  # referred implementation here:
  # https://github.com/tamland/non-linear-optimization
  l = 0.0
  h = 1.0
  for i in range(iters):
    if strong_wolfe_alpha(f, g, alpha, c2):
      return alpha

    half = (l + h) / 2
    alpha = - g_alpha(l) * (h**2) / (2 * (f_alpha(h) - f_alpha(l) - g_alpha(l) * h))
    if alpha < l or alpha > h:
      alpha = half
    if g_alpha(alpha) > 0:
      h = alpha
    elif g_alpha(alpha) <= 0:
      l = alpha
  return alpha

## apply SD, Newton method, and G_N method

In [ ]:
def gradient_descent(f, g, x0, iterations, error):
    xk = x0
    c2 = 0.9
    X,Y=[],[]

    for i in range(iterations):
      # compute search direction
      gk =  g(xk)
      pk = -gk
      if np.linalg.norm(gk) < error:
         break

      # obtain step length by line search
      alpha = step_length(f, g, xk, 1.0, pk, c2)

      # update x
      xk1 = xk + alpha * pk
      X.append(xk[0]), Y.append(xk[1])
      # if np.linalg.norm(xk1 - xk) < error:
      #   xk = xk1
      #   break

      xk = xk1

    #visutization method

    fun_plot(f, 'gradient_descent',x0)
    plt.plot(X,Y, c='black', marker= 'o')
    return xk, i + 1

In [ ]:
def newton(f, g ,h, x0, iterations, error):
    xk = x0
    c2 = 0.9
    X,Y=[],[]

    for i in range(iterations):
      # compute search direction
      gk =  g(xk)
      Hk =  h(xk)

      pk = -Hk.dot(gk)
      if np.linalg.norm(gk) < error:
          break

      # obtain step length by line search
      alpha = step_length(f, g, xk, 1.0, pk, c2)
      # update x
      xk1 = xk + alpha * pk

      X.append(xk[0]), Y.append(xk[1])
     # print('np.linalg.norm(xk1 - xk) ', np.linalg.norm(xk1 - xk), 'and error is ' , error)

      # if np.linalg.norm(xk1 - xk) < error:
      #    xk = xk1
      #    break


      Hk= h(xk1)
      xk = xk1

    #visutization method

    fun_plot(f, 'newton method',x0)
    plt.plot(X,Y, c='black', marker= 'o')
    return xk, i + 1

In [ ]:
#(expressing Rosenbrock as a sum of squares)

# Residuals for Gauss-Newton
def rosenbrock_residuals(x):
    r1 = 10 * (x[1] - x[0]**2)
    r2 = 1 - x[0]
    return np.array([r1, r2])

# Jacobian of the Rosenbrock residuals
def jacobian_rosenbrock(x):
    dr1_dx = -20 * x[0]
    dr1_dy = 10
    dr2_dx = -1
    dr2_dy = 0
    return np.array([[dr1_dx, dr1_dy],
                     [dr2_dx, dr2_dy]])

In [ ]:
#(expressing EG2 as a sum of squares)
# Residuals for Gauss-Newton
def EG2_residuals(x):
    r1 = np.sqrt(2) * np.sin(x[0]**2 + x[0] - 1)
    r2 = np.sin(x[1]**2)
    return np.array([r1, r2])

def jacobian_EG2(x):
    J = np.zeros((2,2))

    # r1 derivatives
    u = x[0]**2 - x[0] - 1
    J[0,0] = np.sqrt(2) * np.cos(u) * (2*x[0] - 1)
    J[0,1] = 0

    # r2 derivatives
    J[1,0] = 0
    J[1,1] = 2 * x[1] * np.cos(x[1]**2)

    return J

In [ ]:
'''
# Gauss-Newton algorithm
def gauss_newton(f, g , x0, iterations, error):
    xk= x0
    X,Y= [],[]
    for i in range(iterations):
        # Calculate residuals and Jacobian
        residuals = EG2_residuals(xk)
        J = jacobian_EG2(xk)
        gk =  g(xk)
        if np.linalg.norm(gk) < error:
            break

        # Gauss-Newton update direction
        pk = - np.linalg.inv(J.T @ J) @ J.T @ residuals

        # Wolfe line search to determine step size alpha_k
        alpha = step_length(f, g, xk,1.0, pk,c2= 0.9)

        # Update the parameter estimate
        xk1 = xk + alpha* pk
        X.append(xk[0]), Y.append(xk[1])

        xk = xk1

    fun_plot(f, 'gauss_newton method',x0)
    plt.plot(X,Y, c='black', marker= 'o')
    return xk, i + 1
    '''

def gauss_newton(f, g, x0, iterations=100, tol=1e-6):
    xk = np.array(x0, dtype=float)

    X, Y = [], []
    lam = 1e-3   # damping factor

    for i in range(iterations):

        r = EG2_residuals(xk)
        J = jacobian_EG2(xk)
        gk = g(xk)

        if np.linalg.norm(gk) < tol:
            break

        # Levenberg-Marquardt modification
        A = J.T @ J + lam * np.eye(len(xk))
        b = J.T @ r

        # solve instead of inverse
        pk = -np.linalg.solve(A, b)

        # line search
        alpha = step_length(f, g, xk, 1.0, pk, c2=0.9)

        xk = xk + alpha * pk

        X.append(xk[0])
        Y.append(xk[1])

        # adaptive damping
        if np.linalg.norm(r) < 1:
            lam *= 0.7
        else:
            lam *= 2

    fun_plot(f, 'Modified Gauss-Newton', x0)
    plt.plot(X, Y, c='black', marker='o')

    return xk, i+1

In [ ]:
def levenberg_marquardt(f, g, x0, iterations, error, mu=1e-3):
    """
    Levenberg-Marquardt algorithm for solving nonlinear least squares problems,
    applied to the Rosenbrock function expressed as a sum of squares.

    Parameters:
        f : callable
            Objective function to minimize, typically sum of squared residuals.
        g : callable
            Gradient of the objective function.
        x0 : ndarray
            Initial guess for the solution.
        iterations : int
            Maximum number of iterations to run.
        error : float
            Tolerance for convergence based on the gradient norm.
        mu : float, optional
            Initial damping parameter (default is 1e-3).

    Returns:
        xk : ndarray
            Estimated solution vector after convergence or max iterations.
        i + 1 : int
            Number of iterations performed.
    """
    xk = x0
    X, Y = [], []

    for i in range(iterations):
        # Compute residuals and Jacobian
        residuals = EG2_residuals(xk)
        J = jacobian_EG2(xk)
        gk = g(xk)

        # stopping condition based on gradient norm
        if np.linalg.norm(gk) < error:
            break

        # Hessian approximation and LM direction
        #LM step: (JᵀJ + μI) d = -Jᵀr
        H = J.T @ J
        A = H + mu * np.eye(len(xk))
        pk = -np.linalg.solve(A, J.T @ residuals)

        # Try step with current direction
        xk1 = xk + pk
        actual_reduction = np.linalg.norm(residuals)**2 - np.linalg.norm(rosenbrock_residuals(xk1))**2

        # Update damping parameter mu
        # Accept step if reduction is sufficie
        if actual_reduction > 0:
            xk = xk1
            mu *= 0.8 # decrease damping
            X.append(xk[0]), Y.append(xk[1])
        else:
            mu *= 2  # increase damping if no improvement

    # visualization
    fun_plot(f, 'levenberg_marquardt', x0)
    plt.plot(X, Y, c='black', marker='o')
    return xk, i + 1

SR1

BFGS and its modified methods

In [ ]:
def SR1(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  for i in range(iterations):
    # compute search direction
    gk =  g(xk)
    pk = -Hk.dot(gk)
    if np.linalg.norm(gk) < error:
        break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = g(xk1)

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # Hs = Hk @ sk
    # if np.dot(yk - Hs, sk) > 1e-8:  # Ensure denominator is not too small
    #     Hk1 = Hk + np.outer((yk - Hs), (yk - Hs)) / np.dot((yk - Hs), sk)


    # compute H_{k+1} by SR1 update
    s_Hy = (sk-(Hk@ yk ))
    if(s_Hy.dot(yk)==0):
        print('divition by zero is not allowed')
        print("Did not converge.")
        break
    rho_k = float(1.0 / s_Hy.dot(yk))

        #rho_k = 1
    Hk1 = Hk+ rho_k* (s_Hy.dot(s_Hy))

    X.append(xk[0]), Y.append(xk[1])

    Hk = Hk1
    xk = xk1

    #visutization method
  fun_plot(f, 'SR1',x0)
  plt.plot(X,Y, c='black', marker= 'o')
  return xk, i + 1

In [ ]:
def Ml_SR1(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  iterations = 9000
  for i in range(iterations):
    # compute search direction
    gk =  g(xk)
    pk = -Hk.dot(gk)
    if np.linalg.norm(gk) < error:
        break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = g(xk1)

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # Hs = Hk @ sk
    # if np.dot(yk - Hs, sk) > 1e-8:  # Ensure denominator is not too small
    #     Hk1 = Hk + np.outer((yk - Hs), (yk - Hs)) / np.dot((yk - Hs), sk)


    # compute H_{k+1} by SR1 update
    s_Hy = (sk- yk)
    if(s_Hy.dot(yk)==0):
        print('divition by zero is not allowed')
        print("Did not converge.")
        break
    rho_k = float(1.0 / s_Hy.dot(yk))

        #rho_k = 1
    Hk1 = I+ rho_k* (s_Hy.dot(s_Hy))

    X.append(xk[0]), Y.append(xk[1])

    Hk = Hk1
    xk = xk1

    #visutization method
  fun_plot(f, 'ML_SR1',x0)
  plt.plot(X,Y, c='black', marker= 'o')
  return xk, i + 1

In [ ]:
def bfgs(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  for i in range(iterations):
    # compute search direction
    gk =  g(xk)
    pk = -Hk.dot(gk)
    if np.linalg.norm(gk) < error:
       break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = g(xk1)

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # compute H_{k+1} by BFGS update
    rho_k = float(1.0 / yk.dot(sk))
    Hk1 = Hk - rho_k*( (np.inner(np.outer(sk,yk), Hk) )+ np.outer(np.inner(Hk,yk) , sk) )+\
        (1+rho_k*(yk.dot(Hk).dot(yk)))*(rho_k* np.outer(sk,sk))

    # Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
    #      rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

    X.append(xk[0]), Y.append(xk[1])

    Hk = Hk1
    xk = xk1
    #visutization method
  fun_plot(f, 'BFGS',x0)
  plt.plot(X,Y, c='black', marker= 'o')

  return xk, i + 1

In [ ]:
def memory_less_bfgs(f, g, x0, iterations, error):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  Hk = I
  X=[]
  Y=[]
  for i in range(iterations):
    # compute search direction
    gk = g(xk)
    pk = -Hk.dot(gk)

    if np.linalg.norm(gk) < error:
      break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = g(xk1)
    X.append(xk[0]), Y.append(xk[1])
    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    # compute H_{k+1} by BFGS update
    rho_k = float(1.0 / yk.dot(sk))

    Hk1 = I - rho_k*( np.outer(sk,yk)+ np.outer(yk,sk) )+\
          (1+rho_k*(yk.dot(yk)))*(rho_k* np.outer(sk,sk))

    # Hk1 = (I - rho_k * np.outer(sk, yk)).dot(I - \
    #         rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)


    # if np.linalg.norm(xk1 - xk) < error:
    #   xk = xk1
    #   break

    Hk = Hk1
    xk = xk1
  fun_plot(f, 'memory less_BFGS',x0)
  plt.plot(X,Y, c='black', marker= 'o')
  return xk, i + 1

In [ ]:
def l_bfgs(f, g, x0, iterations, error, m=10):
  xk = x0
  c2 = 0.9
  I = np.identity(xk.size)
  sks = []
  yks = []
  X=[]
  Y=[]
  def Hp(H0):
    m_t = len(sks)
    q = g(xk)
    a = np.zeros(m_t)
    b = np.zeros(m_t)
    for i in reversed(range(m_t)):
      s = sks[i]
      y = yks[i]
      rho_i = float(1.0 / y.T.dot(s))
      a[i] = rho_i * s.dot(q)
      q = q - a[i] * y

    r = H0.dot(q)

    for i in range(m_t):
      s = sks[i]
      y = yks[i]
      rho_i = float(1.0 / y.T.dot(s))
      b[i] = rho_i * y.dot(r)
      r = r + s * (a[i] - b[i])

    return r

  for i in range(iterations):
    # compute search direction
    gk = g(xk)
    pk = -Hp(I)

    if np.linalg.norm(gk) < error:
       break

    # obtain step length by line search
    alpha = step_length(f, g, xk, 1.0, pk, c2)

    # update x
    xk1 = xk + alpha * pk
    gk1 = g(xk1)

    # define sk and yk for convenience
    sk = xk1 - xk
    yk = gk1 - gk

    sks.append(sk)
    yks.append(yk)
    if len(sks) > m:
      sks = sks[1:]
      yks = yks[1:]

    X.append(xk[0]), Y.append(xk[1])


    # if np.linalg.norm(xk1 - xk) < error:
    #   xk = xk1
    #   break

    xk = xk1
    #visualization L_BFGS method
  fun_plot(f, 'L_BFGS',x0)
  plt.plot(X,Y, c='black', marker='o')
  return xk, i + 1

In [ ]:
def scaled_bfgs(f, g, x0, iterations, error):
    xk = x0
    c2 = 0.9
    I = np.identity(xk.size)
    Hk = I
    X,Y = [],[]

    for i in range(iterations):
    # compute search direction
        gk =  g(xk)
        pk = -Hk.dot(gk)

        if np.linalg.norm(gk) < error:
              break

        # obtain step length by line search
        alpha = step_length(f, g, xk, 1.0, pk, c2)

        # update x
        xk1 = xk + alpha * pk
        gk1 = g(xk1)

        # define sk and yk for convenience
        sk = xk1 - xk
        yk = gk1 - gk

        # compute H_{k+1} by BFGS update
        rho_k = (1.0 / yk.T.dot(sk))

        #Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
         #       rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

        delta = (rho_k*6* (f(xk)-f(xk1)+sk.T.dot(gk1)))-2  #BFGSB
        #delta = (rho_k*2* (f(xk)-f(xk1)+sk.T.dot(gk1)))    ##BFGSY
        #delta = ( yk.T.dot(sk))/ (yk.T.dot(yk)) #rho_k*(yk.T.dot(yk)) BFGSC

        '''
        #BFGSN
        Beta=np.exp(-1/np.power(i,2))
        delta = (yk.T.dot(sk))/ ((yk.T.dot(yk))+Beta)

        if alpha <np.exp(-1/np.power(i,2)):
            alpha = np.exp(-1/np.power(i,2))
            delta = 1
        '''

        Hk1 = Hk -rho_k*( (np.inner(np.outer(sk,yk), Hk) )+ np.outer(np.inner(Hk,yk.T) , sk) )+\
                ((1.0/delta)+rho_k*(yk.T.dot(Hk).dot(yk)))*(rho_k* np.outer(sk,sk))

        '''
        Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
                rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)
        '''
        # if np.linalg.norm(xk1 - xk) < error:
        #   xk = xk1
        #   break

        X.append(xk[0]), Y.append(xk[1])
        Hk = Hk1
        xk = xk1
        #visutization method
    fun_plot(f, 'scaled_bfgs',x0)
    plt.plot(X,Y, c='black', marker= 'o')

    return xk, i + 1


In [ ]:
def double_scaled_bfgs(f, g, x0, iterations, error):
    xk = x0
    c2 = 0.9
    I = np.identity(xk.size)
    Hk = I
    X,Y = [],[]
    for i in range(iterations):
    # compute search direction
        gk =  g(xk)
        pk = -Hk.dot(gk)

        if np.linalg.norm(gk) < error:
              break

        # obtain step length by line search
        alpha = step_length(f, g, xk, 1.0, pk, c2)

        # update x
        xk1 = xk + alpha * pk
        gk1 = g(xk1)

        # define sk and yk for convenience
        sk = xk1 - xk
        yk = gk1 - gk

        # compute H_{k+1} by BFGS update
        rho_k = (1.0 / yk.T.dot(sk))

        #Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
         #       rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)

        delta = (yk.T.dot(sk))/ ((yk.T.dot(yk))+(sk.T.dot(gk1)))
        alpha =(2-(delta*(yk.T.dot(yk)/yk.T.dot(sk))))/1

        delta = 1.0/delta
        alpha = 1.0/alpha

        Hk1 = alpha*I - alpha* (rho_k*( np.outer(sk,yk)+ np.outer(yk,sk)))+\
                    (delta+ alpha*rho_k*(yk.dot(yk)))*(rho_k* np.outer(sk,sk))
        '''
        Hk1 = (I - rho_k * np.outer(sk, yk)).dot(Hk).dot(I - \
                rho_k * np.outer(yk, sk)) + rho_k * np.outer(sk, sk)
        '''
        # if np.linalg.norm(xk1 - xk) < error:
        #   xk = xk1
        #   break

        X.append(xk[0]), Y.append(xk[1])
        Hk = Hk1
        xk = xk1
        #visutization method
    fun_plot(f, ' double_scaled_bfgs',x0)
    plt.plot(X,Y, c='black', marker= 'o')

    return xk, i + 1


## make initialization

In [ ]:
x0 = np.array([1,1])
error = 1e-5
max_iterations = 1000

#plot objective function for visualization
plot_3D(fun, x0)
fun_plot(fun, '41. EG2(CUTE)',x0)

# apply methods

In [ ]:
print( '\n======= symmetric rank one=====\n')
start = time.time()
x, n_iter = SR1(fun, grad, x0,
                iterations=max_iterations, error=error)
end = time.time()
print ("  SR1 terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
  .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print('  SR1 accuracy',accuracy(fun(x)))

In [ ]:
print( '\n======= Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter = bfgs(fun, grad, x0,
                iterations=max_iterations, error=error)
end = time.time()
print ("  BFGS terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
  .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print('  BFGS accuracy',accuracy(fun(x)))


In [ ]:
print( '\n======= Memory less_symmetric rank one=====\n')
start = time.time()
x, n_iter = Ml_SR1(fun, grad, x0,
                iterations=max_iterations, error=error)
end = time.time()
print (" Ml_SR1 terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
  .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print(' Ml_SR1 accuracy',accuracy(fun(x)))

In [ ]:
print( '\n======= Memory Less Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter = memory_less_bfgs(fun, grad, x0,
                iterations=max_iterations, error=error)
end = time.time()
print ("  Memory_Less BFGS terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
        .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print('  Memory_Less BFGS accuracy',accuracy(fun(x)))

In [ ]:
print ('\n======= Limited memory Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter = l_bfgs(fun, grad, x0,iterations=max_iterations, error=error)
end = time.time()
print ("  l-BFGS terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
    .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print('  l-BFGS accuracy',accuracy(fun(x)))

In [ ]:
print( '\n======= Scaled Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter = scaled_bfgs(fun, grad, x0, max_iterations, error)

end = time.time()
print ("  scaled_bfgs terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
        .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print('  scaled_bfgsS accuracy ',accuracy(fun(x)))
#print("{:.3f}".format(num))

#BFGSB
scaled_bfgs terminated in 17 iterations,
x = [1.00000016 1.00000031],
 f(x) = 3.359523234777152e-14,
  time elapsed 0.3771049976348877,
   time per iter 0.022182646919699275
  scaled_bfgsS accuracy  0 999999999999664

#BFGSY

 terminated in 21 iterations,

  x = [0.99999997 0.99999995],
  
   f(x) = 2.617652659639188e-15,
   
time elapsed 0.21242046356201172, time per iter 0.010115260169619606
    
accuracy  0.9999999999999973

#BFGSC
 terminated in 1000 iterations,

  x = [nan nan],

  f(x) = nan,
   time elapsed 1.0225350856781006, time per iter 0.0010225350856781007

  scaled_bfgsc accuracy  nan

#BFGS scaled where delta = (rho_k*(yk.T.dot(yk)))

terminated in 1000 iterations,

 x = [1.00195156 1.00287004],

  f(x) = 0.00011132298032779838,
  
  time elapsed 1.2167713642120361, time per iter 0.0012167713642120361
  
   accuracy  0.9998886770196722

In [ ]:
print( '\n======= Double Scaled Broyden-Fletcher-Goldfarb-Shanno ======\n')
start = time.time()
x, n_iter =  double_scaled_bfgs(fun, grad, x0, max_iterations, error)

end = time.time()
print ("  double_scaled_bfgs terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
        .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))

print(' double_scaled_bfgs accuracy ',accuracy(fun(x)))
#print("{:.3f}".format(num))

In [ ]:
print('\n======= Bgradient_descent======\n')
start = time.time()
x, n_iter = gradient_descent(fun, grad, x0,
                 iterations=max_iterations, error=error)
end = time.time()
print("  gradient_descent terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
    .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print('  Bgradient_descent accuracy', accuracy(fun(x)))



In [ ]:
print('\n======= Newton Method ======\n')
start = time.time()
x, n_iter = newton(fun, grad, hess, x0,
iterations=max_iterations, error=error)
end = time.time()
print(" Newton terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"
.format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print(' Newton accuracy', accuracy(fun(x)))

In [ ]:
print( '\n======= Gauss newton method  ======\n')
start = time.time()
x, n_iter = gauss_newton(fun, grad, x0,
                  iterations=max_iterations)
end = time.time()
print ("  GN terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
   .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print('  GN accuracy', accuracy(fun(x)))

In [ ]:
print( '\n======= Levenberg_Marquardt method  ======\n')
start = time.time()
x, n_iter = levenberg_marquardt(fun, grad, x0,
                  iterations=max_iterations, error= error)
end = time.time()
print ("  LM terminated in {} iterations, x = {}, f(x) = {}, time elapsed {}, time per iter {}"\
   .format(n_iter, x, fun(x), end - start, (end - start) / n_iter))
print('  LM accuracy', accuracy(fun(x)))



In [ ]:
import numpy as np

# =========================
# Function
# =========================
def f(x):
    n = len(x)
    total = 0.0

    for i in range(n-1):
        total += np.sin(x[0] + x[i]**2 - 1)

    total += 0.5 * np.sin(x[-1]**2)

    return total

# =========================
# Gradient
# =========================
def grad(x):
    n = len(x)
    g = np.zeros(n)

    # derivative w.r.t x1
    for i in range(n-1):
        g[0] += np.cos(x[0] + x[i]**2 - 1)

    # derivatives w.r.t x2 ... x(n-1)
    for j in range(1, n-1):
        g[j] = 2 * x[j] * np.cos(x[0] + x[j]**2 - 1)

    # derivative w.r.t xn
    g[-1] = x[-1] * np.cos(x[-1]**2)

    return g

# =========================
# Hessian
# =========================
def hessian(x):
    n = len(x)
    H = np.zeros((n, n))

    # First row & column (coupling with x1)
    for i in range(n-1):
        val = -np.sin(x[0] + x[i]**2 - 1)

        # diagonal x1,x1
        H[0,0] += val

        if i != 0:
            H[0,i] = 2*x[i]*val
            H[i,0] = H[0,i]

    # diagonal for x2 ... x(n-1)
    for j in range(1, n-1):
        u = x[0] + x[j]**2 - 1
        H[j,j] = 2*np.cos(u) - 4*(x[j]**2)*np.sin(u)

    # xn part
    xn = x[-1]
    H[-1,-1] = np.cos(xn**2) - 2*(xn**2)*np.sin(xn**2)

    return H

In [ ]:
x = np.ones(5)  # x0 = [1,1,...,1]

print("f(x) =", f(x))
print("grad =", grad(x))
print("Hessian =\n", hessian(x))